# Step 0 · Data preparation

1. Run sections 1–2.1 on TIP. They search PATSTAT, keep one row per application, and write the scorable applications and an input receipt under the active run's `preparation/` directory.
2. Transfer **both** the Parquet and its adjacent `.receipt.json` to the approved scoring environment. Run `1c-SbertScoring_Colab.ipynb`. Copy its returned scored Parquet and `.handoff.json` into the run's `colab_return/` staging directory.
3. On TIP, continue from section 2.2. After a kernel restart run the imports and settings first. The notebook verifies the transfer and publishes the selected applications and `manifest.json`. Run `2c-ProdValidation.ipynb` on the same run for precision and recall.

Query development is documented in `1a-QueryDiagnostics.ipynb`. The TEST enrichment experiment is documented in `validation/enrichment_decision.csv`.


In [1]:
%env PATENT_RUN_ID=phase2a-prod-001

env: PATENT_RUN_ID=phase2a-prod-001


In [2]:
from datetime import date

import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [3]:
from pathlib import Path
import hashlib
import json
import shutil
import sys

import numpy as np
import pyarrow.parquet as pq

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run_baseline.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from run_baseline import Run


In [4]:
r = Run.open(ROOT)
ENV = r.env
YEARS = (r.cfg["years"]["query_min"], r.cfg["years"]["query_max"])
INPUT_REL = "preparation/df_sbert_input.parquet"
SCORED_REL = "preparation/df_sbert_scored.parquet"
SELECTED_REL = "preparation/df_selected.parquet"
RETURN_DIR = r.path("colab_return/df_sbert_scored.parquet").parent
TAU_FILTER = 0.45
RANDOM_STATE = 42

DRONE_DESCRIPTION = """
Flying drones: aircraft that fly without a human pilot on board, remotely
piloted or autonomous, of any size, configuration and end use.

In scope:
1. The vehicle: airframe, wings and rotors, propulsion and power supply,
   undercarriage.
2. Flight control, navigation, obstacle avoidance, command and data links,
   ground control stations, swarm coordination.
3. Dedicated infrastructure: launch, landing, recovery, charging or battery
   swap, docking, storage and transport.
4. Airspace and traffic management for unmanned aircraft.
5. Mission equipment and applications, when the invention concerns the
   aircraft itself or the way it is operated in flight (e.g. payload release
   for delivery, spraying controlled through the flight path).

Out of scope:
- Generic technologies for which a drone is only one of several possible
  carriers or uses.
- Inventions that only use data or services provided by a drone (e.g.
  analysis of images taken from a drone).
- Aircraft with a pilot on board.
- Ground-based counter-drone systems (a drone that intercepts other drones
  is in scope).
- Underwater, surface and ground unmanned vehicles.
- Guided missiles and projectiles.

Test for doubtful cases: if the drone is removed, does the invention still
make sense? If yes, it is out of scope.
""".strip()

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"

# The positive part of the sector definition (first paragraph + "In scope"):
# an embedding model does not understand negation, so the "Out of scope" list
# of DRONE_DESCRIPTION is left out.
SBERT_DESCRIPTION = """
Flying drones: aircraft that fly without a human pilot on board, remotely
piloted or autonomous, of any size, configuration and end use.

In scope:
1. The vehicle: airframe, wings and rotors, propulsion and power supply,
   undercarriage.
2. Flight control, navigation, obstacle avoidance, command and data links,
   ground control stations, swarm coordination.
3. Dedicated infrastructure: launch, landing, recovery, charging or battery
   swap, docking, storage and transport.
4. Airspace and traffic management for unmanned aircraft.
5. Mission equipment and applications, when the invention concerns the
   aircraft itself or the way it is operated in flight (e.g. payload release
   for delivery, spraying controlled through the flight path).
""".strip()

# Liste definitive. Come sono state scelte: Step0/1a-QueryDiagnostics.ipynb

DRONE_KEYWORDS = (
    # Names of the vehicle
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "unmanned air vehicle",
    "unmanned helicopter",
    "uncrewed aircraft",
    "remotely piloted aircraft",

    # Configurations
    "quadcopter",
    "quadrotor",
    "quadrocopter",
    "multicopter",
    "hexacopter",
    "octocopter",
    "micro air vehicle",
)

# Cercate come parola intera (espressione regolare), non come pezzo di testo:
# "drone" come sottostringa trovava anche "dronedarone", "dihydronepetalactone", ...
DRONE_KEYWORD_PATTERNS = (
    r"\bdrones?\b",
)

DRONE_IPC_CODES = (
    "B64U",
    "G08G5/57",
)

DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
    "G08G5/57",
)

def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

methodology = {
    "query_years": list(YEARS),
    "years": dict(r.cfg["years"]),
    "enrichment": False,
    "keywords_substring": list(DRONE_KEYWORDS),
    "keyword_patterns": list(DRONE_KEYWORD_PATTERNS),
    "ipc_prefixes": list(DRONE_IPC_CODES),
    "cpc_prefixes": list(DRONE_CPC_CODES),
    "retrieval": "OR union of distinct application IDs; filing year inclusive",
    "representative_application_text": "English title and longest English abstract per application",
    "eligible": "Nonempty English abstract",
    "sector_description": DRONE_DESCRIPTION,
    "sector_description_sha256": text_sha256(DRONE_DESCRIPTION),
    "sbert_model": SBERT_MODEL,
    "sbert_description": SBERT_DESCRIPTION,
    "sbert_description_sha256": text_sha256(SBERT_DESCRIPTION),
    "sbert_text": "title + '. ' + abstract",
    "sbert_scoring": "cosine (normalized dot product), identical texts encoded once",
    "threshold": TAU_FILTER,
    "selection": "keep application when score >= threshold; any passing member selects the family",
    "random_state": RANDOM_STATE,
}


## Sector definition

Flying drones are aircraft that fly without a human pilot on board, remotely piloted or autonomous, of any size, configuration and end use.

| Area | In scope |
|---|---|
| **Vehicle** | Airframe, wings and rotors, propulsion and power supply, undercarriage |
| **Control and communications** | Flight control, navigation, obstacle avoidance, command and data links, ground control stations, swarms |
| **Dedicated infrastructure** | Launch, landing, recovery, charging or battery swap, docking, storage and transport |
| **Airspace management** | Traffic management for unmanned aircraft |
| **Mission equipment and applications** | Only when the invention concerns the aircraft itself or the way it is operated in flight |

Out of scope: generic technologies for which a drone is one of several possible uses; inventions that only use data or services provided by a drone; aircraft with a pilot on board; ground-based counter-drone systems; underwater, surface and ground unmanned vehicles; guided missiles and projectiles.

**Rule for doubtful cases:** if the drone is removed, does the invention still make sense? If yes, it is out of scope.

The full text is `DRONE_DESCRIPTION` above: it is the definition given to the LLM validator and to the human labellers. The similarity is computed against a shorter, positive-only version, `SBERT_DESCRIPTION`, transferred to Colab in the verified input receipt.


In [5]:
patstat = PatstatClient(env=ENV) # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()  # SQLAlchemy session used by all ORM queries below

## 1. Search: keywords, IPC and CPC

Final lists. Codes are matched by prefix (`B64U` selects the whole subclass). Keywords are matched as substrings of titles and abstracts, except "drone", which is matched as a whole word.

In [6]:
# The final search criteria are in the settings cell for both parts of the run.
print("CPC:", DRONE_CPC_CODES, "| IPC:", DRONE_IPC_CODES)


CPC: ('B64U', 'B64C39/024', 'G08G5/57') | IPC: ('B64U', 'G08G5/57')


In [7]:
def keyword_condition(terms, patterns=()):
    """OR-condition: a term (substring) or a pattern (regular expression)
    appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
        *(title.regexp_match(p) for p in patterns),
        *(abstract.regexp_match(p) for p in patterns),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)

#3 TYPES OF QUERIES

#PROF BASELINE, MODIFIES:
#in each -> .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))

def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS, DRONE_KEYWORD_PATTERNS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.

appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

#each query (keyword,CPC,IPD) retrieve only appln_id foreach application

#with this function we retrieve all the others attributes
def query_details(ids_table):
    return (
        db.query(
            *APPLN_COLUMNS,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
        )
        .select_from(TLS201_APPLN)
        .join(
            ids_table,
            TLS201_APPLN.appln_id == ids_table.c.appln_id,
        )
        .outerjoin(
            TLS202_APPLN_TITLE,
            and_(
                TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
                func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en",
            ),
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            and_(
                TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
                func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
            ),
        )
        .distinct()
    )

# Prepare the DataFrame with one row per patent application (appln_id).
# Replace missing titles and abstracts with empty strings.
# If multiple rows exist for the same application, keep the longest abstract.
def one_application(df):
    columns = [c.key for c in APPLN_COLUMNS]
    columns += ["appln_title", "appln_abstract"]
    df = df.reindex(columns=columns).copy()

    for column in ["appln_title", "appln_abstract"]:
        df[column] = df[column].fillna("").astype(str).str.strip()

    df["_abstract_length"] = df.appln_abstract.str.len()

    return (
        df.sort_values(
            ["appln_id", "_abstract_length", "appln_abstract", "appln_title"],
            ascending=[True, False, True, True],
        )
        .drop_duplicates("appln_id")
        .drop(columns="_abstract_length")
        .reset_index(drop=True)
    )

q_drone = query_details(drone_ids)
df_drone = one_application(patstat.df(q_drone))

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,2442,38192468,EP,06838485,A,2006-11-29,2005,Y,ELECTRIC MOTOR ASSISTED TAKEOFF DEVICE FOR AN ...,"A takeoff assist device for an air vehicle, su..."
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
2,28800,39609207,WO,2007069569,W,2007-05-23,2006,N,DUAL-USE MODULAR PROPULSION SURVEILLANCE VEHIC...,The present invention provides a system for re...
3,28822,39609212,WO,2007073846,W,2007-07-19,2006,N,AERODYNAMIC INTEGRATION OF A PAYLOAD CONTAINER...,A vertical takeoff and landing (VTOL) rotary-w...
4,36005,37810003,WO,2008000133,W,2008-01-16,2007,N,APPARATUS FOR A HIVE,A trap(1) for attachment to a hive entrance co...
5,39269,39636744,WO,2008051476,W,2008-01-18,2007,N,ROTARYCRAFT POWER AND PROPULSION SYSTEM,A rotary wing vehicle includes a body structur...
6,52470,38535646,EP,08101036,A,2008-01-29,2007,Y,Method and system for securely authorising the...,The system has a ground control station (100) ...
7,67606,39185865,FR,0608714,A,2006-10-03,2006,N,Electronic telemetry device for e.g. minidrone...,The device has a miniaturized controller (1) t...
8,68294,40210552,FR,0853393,A,2008-05-26,2008,Y,,
9,76326,39666849,WO,2007021962,W,2007-10-15,2006,N,LENTICULAR AIRSHIP,An airship (10) may include a hull (22) substa...


## 2. SBERT filter

Each application with an English abstract is compared with a short description of the sector; the cosine similarity is its score, and applications below `TAU_FILTER` are excluded.

Two choices follow from the validation of the first version of the filter: the text compared is **title + abstract** (many relevant applications name the drone only in the title), and the description contains **no exclusions** (an embedding model does not understand negation, so naming excluded technologies makes them more similar, not less).

### 2.1 Export for SBERT (end of part A)

Writes `df_sbert_input.parquet`: the applications that SBERT can score, with all their columns. Download it with its input receipt from the run directory and run `1c-SbertScoring_Colab.ipynb` on Colab.

Applications without an English abstract cannot be scored and are not part of the dataset: the cell prints how many they are, and how many families have no scorable application at all.


In [8]:
#SECTION 2 : SBERT AS FILTER

# Applications with a non-empty English abstract: the only ones SBERT can score.
df_sbert = df_drone[
    df_drone["appln_abstract"].fillna("").str.strip().ne("")
].copy()

print("Applications with an abstract:", len(df_sbert))
print("Applications without an abstract:", len(df_drone) - len(df_sbert))

# Families that cannot be evaluated: none of their applications has an English abstract.
families_without_text = set(df_drone["docdb_family_id"]) - set(df_sbert["docdb_family_id"])
print("Families extracted:", df_drone["docdb_family_id"].nunique())
print("Families with no English abstract (not evaluated):", len(families_without_text))

input_metadata = {
    "methodology": methodology,
    "retrieved_applications": len(df_drone),
    "retrieved_families": int(df_drone["docdb_family_id"].nunique()),
    "scorable_applications": len(df_sbert),
    "families_without_english_abstract": len(families_without_text),
}
input_file = r.publish(INPUT_REL, lambda path: df_sbert.to_parquet(path, index=False), metadata=input_metadata)
RETURN_DIR.mkdir(parents=True, exist_ok=True)
print("Send to Colab:", input_file, "and", str(input_file) + ".receipt.json")
print("Stage both returned files in:", RETURN_DIR)


Applications with an abstract: 216775
Applications without an abstract: 12281
Families extracted: 186124
Families with no English abstract (not evaluated): 4401
Send to Colab: /home/jovyan/flying-drones-phase2a/outputs/runs/PROD/phase2a-prod-001/preparation/df_sbert_input.parquet and /home/jovyan/flying-drones-phase2a/outputs/runs/PROD/phase2a-prod-001/preparation/df_sbert_input.parquet.receipt.json
Stage both returned files in: /home/jovyan/flying-drones-phase2a/outputs/runs/PROD/phase2a-prod-001/colab_return


### 2.2 Similarity scores (start of part B)

Import the Colab return from the active run's `colab_return/` directory. The handoff binds the scores to the input bytes, run and scoring definition. The import also checks every original application row. Subsequent reads use the verified, published copy.


In [9]:
input_file = r.verify(INPUT_REL)
input_receipt = r.receipt(INPUT_REL)
if input_receipt["metadata"]["methodology"] != methodology:
    raise ValueError("Run settings differ from the sealed SBERT input. Choose the correct run.")

def load_scores(scored_file, handoff):
    expected_lineage = {
        "input_sha256": input_receipt["sha256"],
        "input_receipt_sha256": file_sha256(Path(str(input_file) + ".receipt.json")),
        "env": ENV, "run_id": r.run_id,
        "config_sha256": input_receipt["config_sha256"], "sbert_model": SBERT_MODEL,
        "sbert_description_sha256": text_sha256(SBERT_DESCRIPTION),
    }
    if any(handoff.get(key) != value for key, value in expected_lineage.items()):
        raise ValueError("Scoring receipt does not match the run, data or scoring definition.")
    if file_sha256(scored_file) != handoff.get("scored_sha256"):
        raise ValueError("Returned scored Parquet digest differs from the scoring receipt.")
    scored = pd.read_parquet(scored_file)
    original = pq.ParquetFile(input_file)
    columns = original.schema_arrow.names
    if list(scored.columns) != [*columns, "sbert_similarity"] or len(scored) != original.metadata.num_rows:
        raise ValueError("Scoring changed the application schema or row count.")
    if not scored["appln_id"].is_unique or not np.isfinite(scored["sbert_similarity"]).all():
        raise ValueError("Duplicate application or missing/nonfinite score in the scoring return.")
    offset = 0
    for batch in original.iter_batches(batch_size=8192):
        stop = offset + batch.num_rows
        pd.testing.assert_frame_equal(
            scored.iloc[offset:stop][columns].reset_index(drop=True), batch.to_pandas(), check_exact=True
        )
        offset = stop
    return scored

if r.path(SCORED_REL).exists():
    scored_file = r.verify(SCORED_REL)
    handoff = r.receipt(SCORED_REL)["metadata"]["handoff"]
    df_sbert = load_scores(scored_file, handoff)
else:
    returned = RETURN_DIR / "df_sbert_scored.parquet"
    handoff = json.loads((RETURN_DIR / "df_sbert_scored.parquet.handoff.json").read_text("utf-8"))
    df_sbert = load_scores(returned, handoff)

    def copy_scored(path):
        shutil.copyfile(returned, path)
        if file_sha256(path) != handoff["scored_sha256"]:
            raise ValueError("Staged scoring return changed during import.")

    scored_file = r.publish(SCORED_REL, copy_scored, inputs=(INPUT_REL,), metadata={"handoff": handoff})

print("Applications with a score:", len(df_sbert))
df_sbert[["appln_id", "appln_title", "sbert_similarity"]].sort_values("sbert_similarity", ascending=False).head(10)


Applications with a score: 216775


,appln_id,appln_title,sbert_similarity
75431,526116352,COMBINATION OF UNMANNED AERIAL VEHICLES AND TH...,0.971339
30810,487106099,Flying system adopting large unmanned aerial v...,0.967252
146566,583001234,CONTROL SYSTEM AND METHOD FOR PREVENTING FLYIN...,0.966443
168490,600769122,UNMANNED AIRCRAFT SUPPORT DEVICE AND UNMANNED ...,0.966396
18130,473511459,Convertible unmanned aerial vehicle of flight ...,0.965802
106399,550774678,AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED AIR...,0.965671
106400,550774723,AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED AIR...,0.965671
21880,477951970,/ AERIAL LAUNCH AND/OR RECOVERY FOR UNMANNED A...,0.965240
33380,489766292,Unmanned aerial vehicle flying system,0.965051
98093,544991619,Miniature aerial distribution type unmanned ae...,0.964581


### 2.3 Choice of the threshold

The table shows, for each candidate threshold, the share of relevant families kept, the share of non-relevant families removed and the precision of what is kept, on the families already labelled for the current sector definition.

Read it before fixing `TAU_FILTER` in the settings: choose the highest threshold that still keeps almost all relevant families. The labelled families come from samples drawn by group in `1a-QueryDiagnostics`, where "drone" is over-represented, so the table shows how well the score separates the two groups; it is not the precision of the dataset, which is measured in `2c-ProdValidation`.

If the two shares move together at every threshold, the score does not separate relevant from non-relevant families, and the filter removes drones without removing noise.

In [14]:
%pip install "langchain-core==1.7.6" "langchain-openai==1.7.4" "pydantic==2.13.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.5/580.5 kB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 129.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.2/809.2 kB 52.1 MB/s  0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10━━━━━━━━━━━━━━━━━━━━━━  7/16 [idna]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [langchain-openai][langchain-core]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# CALIBRATION OF TAU_FILTER
# Uses the families already labelled for the current sector definition
# (validation/llm_cache.jsonl, written by 1a-QueryDiagnostics). It makes no LLM calls.

VALIDATOR = str(ROOT / "Step0" / "2-DataValidation.ipynb")
%run -i "$VALIDATOR"

pid = prompt_id(DRONE_DESCRIPTION)
labels = pd.Series(
    {fam: r["llm_label"] for (fam, p), r in load_llm_cache().items() if p == pid},
    name="label", dtype="object",
)

# A family is kept if at least one of its applications reaches the threshold,
# so the family score is the highest similarity among its applications.
family_score = df_sbert.groupby("docdb_family_id")["sbert_similarity"].max().rename("score")

calibration = pd.concat([family_score, labels], axis=1, join="inner")
calibration = calibration[calibration["label"].isin(["pertinente", "non_pertinente"])]
relevant = calibration["label"].eq("pertinente")
print("Labelled families with a score:", len(calibration), "| relevant:", int(relevant.sum()))

rows = []
for tau in np.arange(0.30, 0.86, 0.05):
    kept = calibration["score"].ge(tau)
    rows.append({
        "tau": round(float(tau), 2),
        "relevant_kept": round((kept & relevant).sum() / max(relevant.sum(), 1), 3),
        "not_relevant_removed": round((~kept & ~relevant).sum() / max((~relevant).sum(), 1), 3),
        "precision_of_kept": round((kept & relevant).sum() / max(kept.sum(), 1), 3),
    })
pd.DataFrame(rows)


Labelled families with a score: 408 | relevant: 364


,tau,relevant_kept,not_relevant_removed,precision_of_kept
0,0.30,0.992,0.136,0.905
1,0.35,0.984,0.182,0.909
2,0.40,0.973,0.250,0.915
3,0.45,0.940,0.318,0.919
4,0.50,0.920,0.386,0.925
5,0.55,0.874,0.523,0.938
6,0.60,0.830,0.591,0.944
7,0.65,0.712,0.750,0.959
8,0.70,0.571,0.773,0.954
9,0.75,0.412,0.886,0.968


### 2.4 Filter

Applies `TAU_FILTER`. `df_selected` is the dataset.

In [11]:
# Select applications whose similarity reaches the filtering threshold.
df_filtered = df_sbert[
    df_sbert["sbert_similarity"] >= TAU_FILTER
].copy()

print("Applications evaluated with SBERT:", len(df_sbert))
print("Applications retained:", len(df_filtered))
print("Applications excluded by the filter:", len(df_sbert) - len(df_filtered))

df_filtered[
    ["appln_id", "appln_title", "sbert_similarity", "appln_abstract"]
].sort_values("sbert_similarity", ascending=False).head(10)

# Scartati dal filtro iniziale
df_rejected = df_sbert[
    df_sbert["sbert_similarity"] < TAU_FILTER
].copy()

Applications evaluated with SBERT: 216775
Applications retained: 199121
Applications excluded by the filter: 17654


In [12]:
# Dataset produced by this notebook: the applications of the keyword/IPC/CPC union
# retained by the SBERT filter.
df_selected = df_filtered.copy()
df_selected["retrieval_source"] = "initial_union"   # without sbert enrichment this column is useless

# Verify that each application appears only once.
assert not df_selected["appln_id"].duplicated().any()

print("Total selected applications:", len(df_selected))
print("DOCDB families:", df_selected["docdb_family_id"].nunique())

Total selected applications: 199121
DOCDB families: 168664


## 3. Output

Publish the selected application rows and the run's selection manifest. Validation and extraction consume the verified artifacts from this run.


In [13]:
selected_file = r.publish(
    SELECTED_REL, lambda path: df_selected.to_parquet(path, index=False), inputs=(SCORED_REL,),
    metadata={"threshold": TAU_FILTER, "unit": "application"},
)
selected_receipt = r.receipt(SELECTED_REL)
release_inputs = (INPUT_REL, SCORED_REL, SELECTED_REL)
manifest = {
    "schema_version": 1,
    "env": ENV, "run_id": r.run_id,
    "created_utc": selected_receipt["created_utc"], "git_commit": selected_receipt["git_commit"],
    "git_dirty": selected_receipt["git_dirty"],
    "patstat_edition": selected_receipt["patstat_edition"], "config_sha256": selected_receipt["config_sha256"],
    "sector_description": DRONE_DESCRIPTION, "threshold": TAU_FILTER,
    "methodology": methodology,
    "inputs": {rel: r.receipt(rel)["sha256"] for rel in release_inputs},
    "scoring": handoff,
    "query_counts": {key: value for key, value in input_receipt["metadata"].items() if key != "methodology"},
    "selected_rows": len(df_selected),
    "selected_families": int(df_selected["docdb_family_id"].nunique()),
}
manifest_file = r.publish(
    "manifest.json", lambda path: path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"),
    inputs=release_inputs,
)
print("Selection manifest:", manifest_file)
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {manifest['selected_families']:,}")


Selection manifest: /home/jovyan/flying-drones-phase2a/outputs/runs/PROD/phase2a-prod-001/manifest.json
Applications: 199,121
DOCDB families: 168,664
